In [41]:
from google.colab import drive
import os
print("Current working directory:", os.getcwd())



Current working directory: /content


In [42]:
import sys

# Add the folder containing the script to the Python path
sys.path.append('/content/drive/MyDrive/Colab Notebooks/TopicsInMathematics/Labs/')

# Now you can import the module from the script
from decision_tree_regression import decisiontree


ModuleNotFoundError: No module named 'decision_tree_regression'

In [43]:
# Define the root mean square error (RMSE) function
def RMSE(ypred, ytrue):
    return np.sqrt(np.sum((ypred - ytrue) ** 2) / ypred.shape[0])

# Define the Decision Tree class
class decisiontree():
    def __init__(self, max_depth=5, current_depth=0, max_features=None):
        # tree structure values
        self.left_tree = None
        self.right_tree = None
        self.max_depth = max_depth
        self.current_depth = current_depth

        self.best_feature_id = 0.
        self.best_gain = 0.
        self.best_split_value = 0.
        self.MSE = 0.  # In regression we use MSE than gini index so we change the cost function
        self.label = None

        # feature values
        self.X = None
        self.y = None
        self.M = 0
        self.N = 0

        self.max_features = max_features

    def fit(self, X, y):
        self.X = X
        self.y = y
        self.M = X.shape[0]
        self.N = X.shape[1]

        if self.max_features == None or self.max_features > self.N:
            self.max_features = self.N
        if self.current_depth < self.max_depth:
            self.MSE = self.MSE_calculation(self.y)  # Changed to MSE calculation
            self.best_feature_id, self.best_gain, self.best_split_value = self.find_best_split()
            if self.best_gain > 0.:
                self.split_trees()

    def split_trees(self):
        self.left_tree = decisiontree(max_depth=self.max_depth, current_depth=self.current_depth + 1)
        self.right_tree = decisiontree(max_depth=self.max_depth, current_depth=self.current_depth + 1)
        best_feature_values = self.X[:, self.best_feature_id]
        left_indices = np.where(best_feature_values < self.best_split_value)
        right_indices = np.where(best_feature_values >= self.best_split_value)

        left_tree_X = self.X[left_indices]
        left_tree_y = self.y[left_indices]
        right_tree_X = self.X[right_indices]
        right_tree_y = self.y[right_indices]

        self.left_tree.fit(left_tree_X, left_tree_y)
        self.right_tree.fit(right_tree_X, right_tree_y)

    def MSE_calculation(self, y):  # Change GINI calculation to MSE calculation
        if y.size == 0 or y is None:
            return 0.
        return np.mean((y - np.mean(y)) ** 2)

    def find_best_split(self):
        best_feature_id = None
        best_gain = 0.
        best_split_value = None
        n_features = np.random.choice(self.N, self.max_features, replace=False)
        for feature_id in n_features:
            current_gain, current_split_value = self.find_best_split_one_feature(feature_id)
            if best_gain < current_gain:
                best_feature_id = feature_id
                best_gain = current_gain
                best_split_value = current_split_value
        return best_feature_id, best_gain, best_split_value

    def find_best_split_one_feature(self, feature_id):
        feature_values = self.X[:, feature_id]
        unique_feature_values = np.unique(feature_values)
        best_gain = 0.
        best_split_value = None

        if len(unique_feature_values) == 1:
            return best_gain, best_split_value
        for fea_val in unique_feature_values:
            left_indices = np.where(feature_values < fea_val)
            right_indices = np.where(feature_values >= fea_val)

            left_tree_X = self.X[left_indices]
            left_tree_y = self.y[left_indices]

            right_tree_X = self.X[right_indices]
            right_tree_y = self.y[right_indices]

            left_MSE = self.MSE_calculation(left_tree_y) ###MSE calculation
            right_MSE = self.MSE_calculation(right_tree_y) #### MSE calculation

            left_n = left_tree_X.shape[0]
            right_n = right_tree_X.shape[0]

            current_gain = self.MSE - left_n / self.M * left_MSE - right_n / self.M * right_MSE
            if best_gain < current_gain:
                best_gain = current_gain
                best_split_value = fea_val
        return best_gain, best_split_value

    def predict(self, X_test):
        n_test = X_test.shape[0]
        y_pred = np.empty(n_test)
        for i in range(n_test):
            y_pred[i] = self.tree_propogation(X_test[i])
        return y_pred

    def tree_propogation(self, feature):
        # This is a leaf
        if self.left_tree is None:
            return np.mean(self.y)
        # Belongs to the left tree
        if feature[self.best_feature_id] < self.best_split_value:
            return self.left_tree.tree_propogation(feature)
        # Belongs to the right tree
        else:
            return self.right_tree.tree_propogation(feature)


In [44]:
import os

folder_path = '/content/drive/MyDrive/Colab Notebooks/TopicsInMathematics/Labs/'

# List all files in the folder
files = os.listdir(folder_path)

# Print the list of files
for file in files:
    print(file)


Digits
airfoil
lab6.md
lab5.ipynb
lab6.ipynb
decision_tree_regression.py


In [45]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [46]:
import numpy as np
#from decision_tree_regression import decisiontree


class GradientBoostingDecisionTree:
    def __init__(self, max_depth=8, n_estimators=10, lr=0.01, max_features=5, delta=1.0):
        self.max_depth = max_depth
        self.n_estimators = n_estimators
        self.lr = lr # Learning rate (shrinkage)
        self.delta = delta # Huber loss delta parameter

        self.gradient_coeff = []
        self.stop = n_estimators

        self.tree0 = decisiontree(max_depth=self.max_depth, max_features=max_features)
        self.trees = [decisiontree(max_depth=self.max_depth, max_features=max_features) for _ in range(self.n_estimators)]

    def fit(self, X, y):
        self.M, self.N = X.shape

        self.tree0.fit(X, y)
        y_pred = self.tree0.predict(X).reshape((self.M, 1))
        residue = y - y_pred

        for idx, itree in enumerate(self.trees):
            if np.linalg.norm(residue) < 1e-4:
                self.stop = idx
                break

            itree.fit(X, residue)
            ipred = itree.predict(X).reshape((self.M, 1))

            alpha = self.huber_gradient_descent(ipred, residue, self.lr, 150)
            self.gradient_coeff.append(alpha)

            y_pred = np.add(y_pred, self.lr * alpha * ipred)
            residue = y - y_pred
            # Update residue based on Huber loss gradient
            residue = self.huber_loss_gradient(residue, self.delta)

    def predict(self, X_test):
        y_pred = self.tree0.predict(X_test).reshape((-1, 1))
        for idx, itree in enumerate(self.trees):
            if idx == self.stop:
                break
            coeff = self.lr * self.gradient_coeff[idx]
            y_pred = np.add(y_pred, coeff * itree.predict(X_test).reshape((-1, 1)))
        return y_pred

    # def mae_gradient_descent():
    def mae_gradient_descent(self, a, b, lr, epochs):
      alpha = np.random.randn(1)[0]
      for epoch in range(epochs):
        grad = np.sum(self.mae_loss_gradient(b - a * alpha) * (-a))
        alpha -= lr * grad
        return alpha
    # def mae_loss_gradient():
    def mae_loss_gradient(self, a):
      return np.where(a < 0, -1, 1)


    def huber_gradient_descent(self, a, b, lr, epochs):
        alpha = np.random.randn(1)[0]
        for epoch in range(epochs):
            # update alpha using the gradient of the Huber loss
            grad = np.sum(self.huber_loss_gradient(b - a * alpha, self.delta) * (-a))
            alpha -= lr * grad
        return alpha

    def huber_loss_gradient(self, a, delta):
        # Gradient of the Huber loss
        return np.where(np.abs(a) <= delta, -a, -delta * np.sign(a))


In [47]:

if __name__ == '__main__':
    import pandas as pd
    directory = '/content/drive/MyDrive/Colab Notebooks/TopicsInMathematics/Labs/'
    X_train = pd.read_csv(directory + 'airfoil/airfoil_self_noise_X_train.csv').values
    y_train = pd.read_csv( directory + '/airfoil/airfoil_self_noise_y_train.csv').values
    X_test  = pd.read_csv(directory + '/airfoil/airfoil_self_noise_X_test.csv').values
    y_test  = pd.read_csv(directory + '/airfoil/airfoil_self_noise_y_test.csv').values


    GBDT = GradientBoostingDecisionTree(n_estimators=200, max_depth=15)
    GBDT.fit(X_train, y_train)

    y_pred = GBDT.predict(X_test)
    print(RMSE(y_pred, y_test))

2.6948956849578676
